In [ ]:
# Colab bootstrap: install requirements and stage the course data next to the notebook
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/griu/ai-ml-finance-hands-on"
    REPO_DIR = Path("/content/ai-ml-finance-hands-on")
    NOTEBOOK_DIR = Path.cwd()
    DATA_TARGET = NOTEBOOK_DIR / "data"
    REQUIREMENTS_FILE = REPO_DIR / "requirements-colab.txt"

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)],
        check=True,
    )

    shutil.copytree(REPO_DIR / "data", DATA_TARGET, dirs_exist_ok=True)
    print(f"Colab environment detected. Data copied to: {DATA_TARGET.resolve()}")
else:
    print("Local environment detected. Colab bootstrap skipped.")


# NB06f — Association Rules and Apriori for Finance

**Role:** Compact-extension  
**Manual section:** 2.3.2.c — Association rules and Apriori  
**Prerequisites:** NB06d (K-Means and PCA) — for general unsupervised context

---

## Purpose of this notebook

This compact-extension notebook introduces association-rule mining with the Apriori algorithm. Unlike supervised models that predict a target variable, association rules discover **co-occurrence patterns** — which products, services or behaviours tend to appear together.

**Dataset:** `dataset_H_financial_product_baskets.csv`

**Finance motivation:** banks and financial institutions routinely ask: "Which products do customers tend to hold together?" The answer informs cross-sell strategies, product bundle design and relationship-deepening campaigns. Association rules provide a structured, quantitative way to answer these questions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

from pathlib import Path
_candidates = [Path('data/processed'), Path('../data/processed')]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Cannot locate data/processed/. Launch from project root or notebooks/.')

import itertools


In [ ]:
baskets = pd.read_csv(DATA_DIR / 'dataset_H_financial_product_baskets.csv')
baskets.head()

### Understanding the dataset

Each row represents **one customer's product holdings** (a "basket"). The columns include:

- **`customer_id`:** unique customer identifier
- **`age_group`:** customer age bracket
- **`income_group`:** customer income bracket
- **Product columns** (binary: 1 = customer holds the product, 0 = does not): these represent financial products such as savings accounts, loans, credit cards, investment products, insurance, etc.

**Business question:** which financial products tend to be held together? Are there combinations that appear more often than we would expect by chance? If so, a product manager could use these patterns to design cross-sell campaigns.

## Key metrics: support, confidence and lift

Before computing anything, we need clear definitions of the three metrics used in association-rule mining.

### Support

**Support** measures how frequently an itemset appears across all customers:

$$\text{Support}(A) = \frac{\text{Number of customers holding all items in A}}{\text{Total number of customers}}$$

A support threshold filters out rare combinations that are not commercially interesting.

### Confidence

**Confidence** measures how often a rule is correct — given that a customer holds products in the antecedent, how often do they also hold the consequent?

$$\text{Confidence}(A \rightarrow B) = \frac{\text{Support}(A \cup B)}{\text{Support}(A)} = P(B \mid A)$$

### Lift

**Lift** compares the observed co-occurrence with what we would expect if the products were independent:

$$\text{Lift}(A \rightarrow B) = \frac{\text{Confidence}(A \rightarrow B)}{\text{Support}(B)}$$

| Lift value | Interpretation |
|-----------|---------------|
| **Lift > 1** | Products appear together **more often** than expected — positive association |
| **Lift = 1** | No association — products are held independently |
| **Lift < 1** | Products appear together **less often** than expected — negative association |

### Worked mini-example

Suppose we have 100 customers. 40 hold a Savings account, 30 hold Insurance, and 20 hold both.

- Support(Savings, Insurance) = 20/100 = 0.20
- Confidence(Savings → Insurance) = 20/40 = 0.50
- Lift = 0.50 / 0.30 = 1.67

The lift of 1.67 means customers with a Savings account are 67% more likely to hold Insurance than a random customer. This is a cross-sell signal.

## Section 1 — Frequent itemsets

Apriori logic begins by identifying product combinations that appear often enough in the customer base.

In [ ]:
transactions = []
product_cols = [c for c in baskets.columns if c not in ['customer_id', 'age_group', 'income_group']]
for _, row in baskets.iterrows():
    items = [c for c in product_cols if row[c] == 1]
    transactions.append(set(items))

def support(itemset):
    return sum(itemset.issubset(t) for t in transactions) / len(transactions)

results = []
for size in [1, 2, 3]:
    for combo in itertools.combinations(sorted(product_cols), size):
        itemset = frozenset(combo)
        sup = support(itemset)
        if sup >= 0.08:
            results.append({'itemset': itemset, 'size': size, 'support': round(sup, 3)})

frequent = pd.DataFrame(results).sort_values(['size', 'support'], ascending=[True, False])
frequent.head(15)

## Section 2 — Build compact association rules

From frequent itemsets, we build rules of the form **A → B** and measure support, confidence and lift.

In [ ]:
full_support = {row['itemset']: row['support'] for _, row in frequent.iterrows()}
rules = []
for itemset, sup in full_support.items():
    if len(itemset) < 2:
        continue
    for r in range(1, len(itemset)):
        for antecedent in itertools.combinations(itemset, r):
            antecedent = frozenset(antecedent)
            consequent = itemset - antecedent
            conf = sup / support(antecedent)
            cons_sup = support(consequent)
            lift = conf / cons_sup
            rules.append({'antecedent': ', '.join(sorted(antecedent)), 'consequent': ', '.join(sorted(consequent)), 'support': round(sup, 3), 'confidence': round(conf, 3), 'lift': round(lift, 3)})

rules_df = pd.DataFrame(rules).sort_values(['lift', 'confidence', 'support'], ascending=False)
rules_df.head(12)

In [ ]:
top_rules = rules_df.head(8).copy()
plt.barh(top_rules['antecedent'] + ' → ' + top_rules['consequent'], top_rules['lift'])
plt.gca().invert_yaxis()
plt.title('Top rules by lift')
plt.xlabel('Lift')
plt.show()

## What you have learned

- Association rules discover **co-occurrence patterns** in product holdings, not causal relationships.
- **Support** measures frequency, **confidence** measures conditional probability, and **lift** measures how much more likely the co-occurrence is than random chance.
- Rules with **high lift** (> 1) suggest genuine positive associations worth investigating commercially.
- The Apriori algorithm efficiently prunes the search space by first identifying frequent itemsets above a minimum support threshold.

### Business interpretation

The rules with the highest lift point to product combinations that co-occur more often than we would expect by chance. A product manager could use these findings to:

- design **cross-sell campaigns** targeting customers who hold one product but not its common companion,
- create **product bundles** that reflect natural customer behaviour,
- identify **under-served combinations** where co-occurrence is low but business logic suggests potential.

**Important caveat:** association does not imply causation. A strong rule tells us that products co-occur, not that selling one will cause the customer to buy the other. Commercial judgement and testing remain essential.

### Diagnose → Transform → Validate → Justify
- **Diagnose:** we want co-occurrence patterns, not a supervised target.
- **Transform:** customer holdings are converted into a binary basket representation.
- **Validate:** support, confidence and lift are checked against interpretable thresholds.
- **Justify:** useful rules should make business sense and be commercially actionable.

---

*This is a compact-extension notebook supporting manual section 2.3.2.c.*